# Comparar final_100 y ModeloV7 en ANPHY-Sleep

EPCTL04 reproduce la prueba exploratoria anterior. EPCTL05 se incorpora con ambos modelos y el protocolo fijados antes de calcular sus resultados. Cada fila de las tablas corresponde a una noche, un candidato a canal ocular y un modelo. La media de F1 asigna el mismo peso a las cinco etapas.

Ejecuta el notebook con el kernel de `.venv`. Los EDF y los hipnogramas están en `data/sleep-anphy/<registro>/`. Para una prueba rápida con el EDF sintético, deja vacía la lista de registros. [ANPHY-Sleep](https://osf.io/r26fh/)

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

import mne
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import Booster
from tensorflow import keras
from sleep_staging.inference.mixed import MixedSleepStagePredictor

REGISTROS = ["EPCTL04", "EPCTL05"]
CANALES_EOG = ["EOG1", "EOG2"]
UNIDAD_EOG = "uV"
CLASES_V7 = np.array(["W", "N1", "N2", "N3", "REM"])

raiz = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "data/models/dart_crnn_mixto/final_100/manifest.json").is_file())
sys.path.insert(0, str(raiz))
from scripts.compare_cassette_hybrid import contextual_embeddings

carpeta_anphy = raiz / "data/sleep-anphy"

### Adaptar el EDF a la entrada común

Se forman Fpz−Cz y Pz−Oz y se remuestrean las señales de 1.000 a 100 Hz con filtrado antialias. La API admite el EMG ausente. El predictor aplica el preprocesamiento y la fusión habituales.

La cabecera identifica los canales oculares como EOG1 y EOG2 con unidad desconocida. Se ejecutan ambos por separado como candidatos a EOG horizontal, suponiendo µV en los dos casos. Las métricas describen la sensibilidad a esa elección. Cada escenario corresponde a la misma persona y a las mismas épocas.

In [2]:
def adaptar_anphy(origen, destino, canal_eog):
    import edfio
    from scipy.signal import resample_poly

    escala_uv = {"uV": 1, "mV": 1_000, "V": 1_000_000}
    if canal_eog not in ("EOG1", "EOG2") or UNIDAD_EOG not in escala_uv:
        raise ValueError("Revisa el canal y la unidad configurados para ANPHY.")

    registro = edfio.read_edf(origen, lazy_load_data=True)

    def leer_100_hz(nombre, unidad):
        canal = registro.get_signal(nombre)
        if canal.sampling_frequency != 1000 or canal.physical_dimension not in (unidad, "?"):
            raise ValueError(f"Revisa la frecuencia y la unidad de {nombre}.")
        return resample_poly(canal.data, up=1, down=10) * escala_uv[unidad]

    senales = {
        "EEG Fpz-Cz": leer_100_hz("FPZ", "uV") - leer_100_hz("CZ", "uV"),
        "EEG Pz-Oz": leer_100_hz("PZ", "uV") - leer_100_hz("OZ", "uV"),
        "EOG horizontal": leer_100_hz(canal_eog, UNIDAD_EOG),
    }
    edfio.Edf([
        edfio.EdfSignal(valores, 100, label=nombre, physical_dimension="uV")
        for nombre, valores in senales.items()
    ]).write(destino)

### Inferencia con el preprocesamiento de cada modelo

**final_100** ejecuta el predictor instalado de la API. **V7** reproduce el experimento 0 de `ModeloV7.ipynb`. Aplica un filtro de 0,3–35 Hz, segmenta en 30 segundos y normaliza cada canal dentro de cada época. Si falta el EMG, usa ceros. Con 15 épocas centradas, la CRNN produce una representación de 64 valores. El LightGBM recibe las representaciones anterior, central y siguiente (192 valores). El generador rellena los extremos con la época central.

Se verificó la igualdad de los pesos de la CRNN y de los árboles del LightGBM entre los archivos de Descargas y las copias compatibles en `data/referencias/hibrido_v6/`. El experimento 0 utiliza el clasificador directamente. El HMM y las características espectrales adicionales pertenecen a las variantes posteriores del notebook del compañero.

Para simular una noche entrante, los dos modelos predicen todas las épocas completas. El notebook original de V7 usaba etiquetas para recortar la vigilia y descartar épocas antes de crear ventanas. Aquí se conserva la continuidad temporal y se leen las etiquetas al puntuar.

In [3]:
def predecir_v7(entrada, extractor, clasificador):
    canales = ["EEG Fpz-Cz", "EEG Pz-Oz", "EOG horizontal", "EMG submental"]
    with mne.io.read_raw_edf(entrada, preload=True, verbose="ERROR") as raw:
        if raw.info["sfreq"] != 100 or not set(canales[:3]).issubset(raw.ch_names):
            raise ValueError("V7 requiere las dos derivaciones EEG y el EOG a 100 Hz.")
        presentes = [c for c in canales if c in raw.ch_names]
        raw.pick(presentes).filter(0.3, 35.0, verbose=False)
        n = raw.n_times // 3000
        epocas = np.zeros((n, 3000, 4), dtype="float32")
        for i, canal in enumerate(presentes):
            senal = raw.get_data(picks=[canal], stop=n * 3000).reshape(n, 3000) * 1e6
            epocas[:, :, i] = (senal - senal.mean(axis=1, keepdims=True)) / (
                senal.std(axis=1, keepdims=True) + 1e-8
            )

    vectores = contextual_embeddings(extractor, epocas)
    if vectores.shape != (n, clasificador.num_feature()):
        raise ValueError("Las características de V7 no coinciden con el LightGBM.")
    probabilidades = clasificador.predict(vectores, num_threads=4)
    if probabilidades.shape != (n, 5) or not np.allclose(probabilidades.sum(axis=1), 1):
        raise ValueError("Salida de clasificación V7 inválida.")
    return pd.DataFrame({
        "inicio_s": np.arange(n) * 30, "etapa": CLASES_V7[probabilidades.argmax(axis=1)],
        "confianza": probabilidades.max(axis=1),
    })

In [4]:
modelo = MixedSleepStagePredictor(raiz / "data/models/dart_crnn_mixto/final_100", require_final=True)
pesos_v7 = raiz / "data/referencias/hibrido_v6"
red_v7 = keras.models.load_model(pesos_v7 / "modelo_crnn_maestro_optimo_v7.keras", compile=False)
extractor = keras.Model(red_v7.inputs, red_v7.get_layer("embeddings_latentes_64d").output)
clasificador = Booster(model_file=str(pesos_v7 / "contextual_192d.txt"))
predicciones = {}

for registro in REGISTROS or ["ejemplo"]:
    for canal in CANALES_EOG if REGISTROS else ["nativo"]:
        with TemporaryDirectory() as temporal:
            if REGISTROS:
                entrada = Path(temporal) / "entrada.edf"
                adaptar_anphy(carpeta_anphy / registro / f"{registro}.edf", entrada, canal)
            else:
                entrada = raiz / "apps/api/tests/fixtures/sample_eeg_60s.edf"
            salida = modelo.predict_edf(entrada)
            propia = pd.DataFrame({"inicio_s": salida.onset_seconds, "etapa": salida.stages,
                                   "confianza": salida.probabilities.max(axis=1)})
            companero = predecir_v7(entrada, extractor, clasificador)
        np.testing.assert_array_equal(propia.inicio_s, companero.inicio_s)
        predicciones[registro, canal, "final_100"] = propia
        predicciones[registro, canal, "V7 (192D)"] = companero
        print(f"{registro} · {canal} · {len(propia)} épocas emparejadas")

EPCTL04 · EOG1 · 762 épocas emparejadas


EPCTL04 · EOG2 · 762 épocas emparejadas


EPCTL05 · EOG1 · 755 épocas emparejadas


EPCTL05 · EOG2 · 755 épocas emparejadas


### Comparar con las etapas anotadas

In [5]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, precision_recall_fscore_support

def evaluar(predicciones, etiquetas, clases):
    # Una marca de luces (L) o el último intervalo incompleto no tienen puntuación.
    comparacion = predicciones.merge(etiquetas, on="inicio_s", how="left", validate="one_to_one")
    if not comparacion["duracion_s"].eq(30).all():
        raise ValueError("Faltan anotaciones de 30 segundos alineadas con las predicciones.")
    puntuadas = comparacion.loc[comparacion["referencia"].isin(clases)]
    real, predicha = puntuadas["referencia"], puntuadas["etapa"]
    precision, sensibilidad, f1, n = precision_recall_fscore_support(
        real, predicha, labels=clases, zero_division=0
    )
    detalle = pd.DataFrame({
        "etapa": clases, "epocas_reales": n,
        "precision": precision, "sensibilidad": sensibilidad, "f1": f1,
    })
    resumen = {
        "epocas_evaluadas": len(real), "exactitud": accuracy_score(real, predicha),
        "f1_macro": f1.mean(), "kappa": cohen_kappa_score(real, predicha, labels=clases),
    }
    return resumen, detalle

if REGISTROS:
    globales, etapas = {}, {}
    for registro in REGISTROS:
        etiquetas = pd.read_csv(carpeta_anphy / registro / f"{registro}.txt", sep=r"\s+", header=None,
                                names=["referencia", "inicio_s", "duracion_s"])
        etiquetas["referencia"] = etiquetas["referencia"].replace({"R": "REM"})
        if not etiquetas["referencia"].isin([*CLASES_V7, "L"]).all():
            raise ValueError(f"Etiquetas desconocidas en {registro}.")
        for canal in CANALES_EOG:
            for nombre in ["final_100", "V7 (192D)"]:
                clave = (registro, canal, nombre)
                globales[clave], etapas[clave] = evaluar(predicciones[clave], etiquetas, CLASES_V7)
    indices = ["registro", "EOG candidato", "modelo"]
    globales = pd.DataFrame.from_dict(globales, orient="index").rename_axis(indices)
    etapas = pd.concat(etapas, names=indices).droplevel(-1)
    display(globales.round(4), etapas.round(4))
else:
    for clave, tabla in predicciones.items():
        print(clave[-1])
        display(tabla)

epocas_evaluadas  exactitud  f1_macro  \
registro EOG candidato modelo                                             
EPCTL04  EOG1          final_100               735     0.8163    0.7583   
                       V7 (192D)               735     0.4762    0.4138   
         EOG2          final_100               735     0.8259    0.7779   
                       V7 (192D)               735     0.4707    0.4057   
EPCTL05  EOG1          final_100               729     0.7229    0.6496   
                       V7 (192D)               729     0.3717    0.3991   
         EOG2          final_100               729     0.7215    0.6484   
                       V7 (192D)               729     0.3745    0.4043   

                                   kappa  
registro EOG candidato modelo             
EPCTL04  EOG1          final_100  0.7541  
                       V7 (192D)  0.3657  
         EOG2          final_100  0.7681  
                       V7 (192D)  0.3620  
EPCTL05  EOG1          final_100  0.6233  
                       V7 (192D)  0.2290  
         EOG2          final_100  0.6228  
                       V7 (192D)  0.2372

etapa  epocas_reales  precision  \
registro EOG candidato modelo                                      
EPCTL04  EOG1          final_100     W            228     0.7362   
                       final_100    N1             95     0.7857   
                       final_100    N2            214     0.8663   
                       final_100    N3            124     0.9593   
                       final_100   REM             74     0.7867   
                       V7 (192D)     W            228     0.9591   
                       V7 (192D)    N1             95     0.9286   
                       V7 (192D)    N2            214     0.5200   
                       V7 (192D)    N3            124     1.0000   
                       V7 (192D)   REM             74     0.1655   
         EOG2          final_100     W            228     0.7724   
                       final_100    N1             95     0.7297   
                       final_100    N2            214     0.8529   
                       final_100    N3            124     0.9516   
                       final_100   REM             74     0.8000   
                       V7 (192D)     W            228     0.9589   
                       V7 (192D)    N1             95     0.7368   
                       V7 (192D)    N2            214     0.7273   
                       V7 (192D)    N3            124     1.0000   
                       V7 (192D)   REM             74     0.1659   
EPCTL05  EOG1          final_100     W             47     0.2733   
                       final_100    N1             71     0.6970   
                       final_100    N2            309     0.8121   
                       final_100    N3            170     0.9829   
                       final_100   REM            132     0.8654   
                       V7 (192D)     W             47     0.6034   
                       V7 (192D)    N1             71     0.5714   
                       V7 (192D)    N2            309     0.4286   
                       V7 (192D)    N3            170     0.9279   
                       V7 (192D)   REM            132     0.2210   
         EOG2          final_100     W             47     0.2733   
                       final_100    N1             71     0.5581   
                       final_100    N2            309     0.8158   
                       final_100    N3            170     0.9746   
                       final_100   REM            132     0.8596   
                       V7 (192D)     W             47     0.5926   
                       V7 (192D)    N1             71     0.6000   
                       V7 (192D)    N2            309     0.8000   
                       V7 (192D)    N3            170     0.9211   
                       V7 (192D)   REM            132     0.2239   

                                  sensibilidad      f1  
registro EOG candidato modelo                           
EPCTL04  EOG1          final_100        0.9912  0.8449  
                       final_100        0.2316  0.3577  
                       final_100        0.8178  0.8413  
                       final_100        0.9516  0.9555  
                       final_100        0.7973  0.7919  
                       V7 (192D)        0.9254  0.9420  
                       V7 (192D)        0.1368  0.2385  
                       V7 (192D)        0.0607  0.1088  
                       V7 (192D)        0.3306  0.4970  
                       V7 (192D)        0.9730  0.2829  
         EOG2          final_100        0.9825  0.8649  
                       final_100        0.2842  0.4091  
                       final_100        0.8131  0.8325  
                       final_100        0.9516  0.9516  
                       final_100        0.8649  0.8312  
                       V7 (192D)        0.9211  0.9396  
                       V7 (192D)        0.1474  0.2456  
                       V7 (192D)        0.0374  0.0711  
                       V7 (192D)    